In [0]:
%sql
-- Create control schema
CREATE SCHEMA IF NOT EXISTS retail_catalog.control;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_catalog.control.processed_files
(
    table_name STRING,
    source_file STRING,
    status STRING,
    processed_at TIMESTAMP,
    records_processed INT
);

In [0]:
%sql
-- Create the silver customers table

CREATE TABLE IF NOT EXISTS retail_catalog.silver.customers
(
    CustomerID STRING,
    CustomerName STRING,
    Region STRING,
    CustomerSegment STRING,
    SignupDate DATE,
    _ingestion_timestamp TIMESTAMP,
    _source_file STRING
);

In [0]:
%sql
-- Create customer incremental view

CREATE OR REPLACE TEMP VIEW customers_incremental AS

WITH ranked_customers AS (
    SELECT
        b.CustomerID,
        TRIM(b.CustomerName) AS CustomerName,
        COALESCE(b.Region, 'Unknown') AS Region,
        TRIM(b.CustomerSegment) AS CustomerSegment,
        b.SignupDate,
        b._ingestion_timestamp,
        b._source_file,

        ROW_NUMBER() OVER (
            PARTITION BY b.CustomerID
            ORDER BY b._ingestion_timestamp DESC
        ) AS rn

    FROM retail_catalog.bronze.customers_raw b

    LEFT ANTI JOIN (
        SELECT source_file
        FROM retail_catalog.control.processed_files
        WHERE table_name = 'customers'
          AND status = 'SUCCESS'
    ) p
    ON b._source_file = p.source_file

    WHERE b.CustomerID IS NOT NULL
)

SELECT
    CustomerID,
    CustomerName,
    Region,
    CustomerSegment,
    SignupDate,
    _ingestion_timestamp,
    _source_file
FROM ranked_customers
WHERE rn = 1;

In [0]:
%sql
-- Perform the merge
MERGE INTO retail_catalog.silver.customers AS tgt
USING customers_incremental AS src

ON tgt.CustomerID = src.CustomerID

WHEN MATCHED THEN
    UPDATE SET
        tgt.CustomerName = src.CustomerName,
        tgt.Region = src.Region,
        tgt.CustomerSegment = src.CustomerSegment,
        tgt.SignupDate = src.SignupDate,
        tgt._ingestion_timestamp = src._ingestion_timestamp,
        tgt._source_file = src._source_file

WHEN NOT MATCHED THEN
    INSERT (CustomerID, CustomerName, Region, CustomerSegment, SignupDate,
    _ingestion_timestamp, _source_file)
    VALUES (
        src.CustomerID,
        src.CustomerName,
        src.Region,
        src.CustomerSegment,
        src.SignupDate,
        src._ingestion_timestamp,
        src._source_file
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
-- Record the processed file and count
INSERT INTO retail_catalog.control.processed_files
(table_name, source_file, status, processed_at, records_processed)
SELECT
    element_at(split(_source_file, '/'), -2) AS table_name,
    _source_file,
    'SUCCESS' AS status,
    current_timestamp() AS processed_at,
    COUNT(*) AS records_processed
FROM customers_incremental
GROUP BY _source_file;

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
-- Create the Silver Products table
CREATE TABLE IF NOT EXISTS retail_catalog.silver.products
(
    ProductID STRING,
    ProductName STRING,
    Category STRING,
    Brand STRING,
    UnitPrice DECIMAL(10,2),
    _ingestion_timestamp TIMESTAMP,
    _source_file STRING
);

In [0]:
%sql
-- Create products_incremental

CREATE OR REPLACE TEMP VIEW products_incremental AS

WITH ranked_products AS (
    SELECT
        b.ProductID,
        TRIM(b.ProductName) AS ProductName,
        COALESCE(TRIM(b.Category), 'Unknown') AS Category,
        b.Brand,
        b.UnitPrice,
        b._ingestion_timestamp,
        b._source_file,

        ROW_NUMBER() OVER (
            PARTITION BY b.ProductID
            ORDER BY b._ingestion_timestamp DESC
        ) AS rn

    FROM retail_catalog.bronze.products_raw b

    LEFT ANTI JOIN (
        SELECT source_file
        FROM retail_catalog.control.processed_files
        WHERE table_name = 'products'
          AND status = 'SUCCESS'
    ) p
    ON b._source_file = p.source_file

    WHERE b.ProductID IS NOT NULL
)

SELECT
    ProductID,
    ProductName,
    Category,
    Brand,
    UnitPrice,
    _ingestion_timestamp,
    _source_file
FROM ranked_products
WHERE rn = 1;

In [0]:
%sql
-- Perform the merge
MERGE INTO retail_catalog.silver.products AS tgt
USING products_incremental AS src

ON tgt.ProductID = src.ProductID

WHEN MATCHED THEN
    UPDATE SET
        tgt.ProductName = src.ProductName,
        tgt.Category = src.Category,
        tgt.Brand = src.Brand,
        tgt.UnitPrice = src.UnitPrice,
        tgt._ingestion_timestamp = src._ingestion_timestamp,
        tgt._source_file = src._source_file

WHEN NOT MATCHED THEN
    INSERT
    (ProductID, ProductName, Category, Brand, UnitPrice,
    _ingestion_timestamp, _source_file)
    VALUES
    (
        src.ProductID,
        src.ProductName,
        src.Category,
        src.Brand,
        src.UnitPrice,
        src._ingestion_timestamp,
        src._source_file
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql

INSERT INTO retail_catalog.control.processed_files
(table_name, source_file, status, processed_at, records_processed)
SELECT
    element_at(split(_source_file, '/'), -2) AS table_name,
    _source_file,
    'SUCCESS' AS status,
    current_timestamp() AS processed_at,
    COUNT(*) AS records_processed

FROM products_incremental
GROUP BY _source_file;

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
-- Create the stores table
CREATE TABLE IF NOT EXISTS retail_catalog.silver.stores
(
    StoreID STRING,
    StoreName STRING,
    City STRING,
    Region STRING,
    StoreType STRING,
    _ingestion_timestamp TIMESTAMP,
    _source_file STRING
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stores_incremental AS

WITH ranked_stores AS (
    SELECT
        b.StoreID,
        TRIM(b.StoreName) AS StoreName,
        TRIM(b.City) AS City,
        COALESCE(TRIM(b.Region), 'Unknown') AS Region,
        COALESCE(TRIM(b.StoreType), 'Unknown') AS StoreType,
        b._ingestion_timestamp,
        b._source_file,

        ROW_NUMBER() OVER (
            PARTITION BY b.StoreID
            ORDER BY b._ingestion_timestamp DESC
        ) AS rn

    FROM retail_catalog.bronze.stores_raw b

    LEFT ANTI JOIN (
        SELECT source_file
        FROM retail_catalog.control.processed_files
        WHERE table_name = 'stores'
          AND status = 'SUCCESS'
    ) p
    ON b._source_file = p.source_file

    WHERE b.StoreID IS NOT NULL
)

SELECT
    StoreID,
    StoreName,
    City,
    Region,
    StoreType,
    _ingestion_timestamp,
    _source_file
FROM ranked_stores
WHERE rn = 1;

In [0]:
%sql
-- Perform the merge

MERGE INTO retail_catalog.silver.stores AS tgt
USING stores_incremental AS src

ON tgt.StoreID = src.StoreID
WHEN MATCHED THEN
    UPDATE SET
        tgt.StoreName = src.StoreName,
        tgt.City = src.City,
        tgt.Region = src.Region,
        tgt.StoreType = src.StoreType,
        tgt._ingestion_timestamp = src._ingestion_timestamp,
        tgt._source_file = src._source_file

WHEN NOT MATCHED THEN
    INSERT
    (StoreID, StoreName, City, Region, StoreType,
    _ingestion_timestamp, _source_file)
    VALUES
    (
        src.StoreID,
        src.StoreName,
        src.City,
        src.Region,
        src.StoreType,
        src._ingestion_timestamp,
        src._source_file
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
-- Record the processed file and count
INSERT INTO retail_catalog.control.processed_files
(table_name, source_file, status, processed_at, records_processed)
SELECT
    element_at(split(_source_file, '/'), -2) AS table_name,
    _source_file,
    'SUCCESS' AS status,
    current_timestamp() AS processed_at,
    COUNT(*) AS records_processed
FROM stores_incremental
GROUP BY _source_file;

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
-- Create the silver sales table
CREATE TABLE IF NOT EXISTS retail_catalog.silver.sales
(
    SaleID STRING,
    CustomerID STRING,
    ProductID STRING,
    StoreID STRING,
    SaleDate DATE,
    Quantity INT,
    UnitPrice DECIMAL(10,2),
    Discount DECIMAL(10,2),
    OrderStatus STRING,
    PaymentMethod STRING,
    _ingestion_timestamp TIMESTAMP,
    _source_file STRING
);

In [0]:
%sql
-- Create sales_incremental
CREATE OR REPLACE TEMP VIEW sales_incremental AS

WITH ranked_sales AS (
    SELECT
        b.SaleID,
        b.CustomerID,
        b.ProductID,
        b.StoreID,
        COALESCE(
        TRY_TO_DATE(b.SaleDate, 'M/d/yyyy'),
        TRY_TO_DATE(b.SaleDate, 'yyyy-MM-dd')
        ) AS SaleDate,
        b.Quantity,
        b.UnitPrice,
        COALESCE(b.Discount, 0) AS Discount,
        TRIM(b.OrderStatus) AS OrderStatus,
        TRIM(b.PaymentMethod) AS PaymentMethod,
        b._ingestion_timestamp,
        b._source_file,

        ROW_NUMBER() OVER (
            PARTITION BY b.SaleID
            ORDER BY b._ingestion_timestamp DESC
        ) AS rn

    FROM (
        -- Remove invalid records first
        SELECT *
        FROM retail_catalog.bronze.sales_raw
        WHERE SaleID IS NOT NULL
          AND Quantity > 0
          AND UnitPrice > 0
    ) b

    LEFT ANTI JOIN (
        SELECT source_file
        FROM retail_catalog.control.processed_files
        WHERE table_name = 'sales'
          AND status = 'SUCCESS'
    ) p
    ON b._source_file = p.source_file
)

SELECT
    SaleID,
    CustomerID,
    ProductID,
    StoreID,
    SaleDate,
    Quantity,
    UnitPrice,
    Discount,
    OrderStatus,
    PaymentMethod,
    _ingestion_timestamp,
    _source_file
FROM ranked_sales
WHERE rn = 1;

In [0]:
%sql

MERGE INTO retail_catalog.silver.sales AS tgt
USING sales_incremental AS src

ON tgt.SaleID = src.SaleID
WHEN MATCHED THEN
    UPDATE SET
        tgt.CustomerID = src.CustomerID,
        tgt.ProductID = src.ProductID,
        tgt.StoreID = src.StoreID,
        tgt.SaleDate = src.SaleDate,
        tgt.Quantity = src.Quantity,
        tgt.UnitPrice = src.UnitPrice,
        tgt.Discount = src.Discount,
        tgt.OrderStatus = src.OrderStatus,
        tgt.PaymentMethod = src.PaymentMethod,
        tgt._ingestion_timestamp = src._ingestion_timestamp,
        tgt._source_file = src._source_file

WHEN NOT MATCHED THEN
    INSERT
    (
        SaleID, CustomerID, ProductID, StoreID, SaleDate, Quantity, 
        UnitPrice, Discount, OrderStatus, PaymentMethod, _ingestion_timestamp,
        _source_file
    )
    VALUES
    (
        src.SaleID,
        src.CustomerID,
        src.ProductID,
        src.StoreID,
        src.SaleDate,
        src.Quantity,
        src.UnitPrice,
        src.Discount,
        src.OrderStatus,
        src.PaymentMethod,
        src._ingestion_timestamp,
        src._source_file
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
9901,0,0,9901


In [0]:
%sql
-- Record the processed Sales files
INSERT INTO retail_catalog.control.processed_files
(table_name, source_file, status, processed_at, records_processed)
SELECT
    element_at(split(_source_file, '/'), -2) AS table_name,
    _source_file,
    'SUCCESS' AS status,
    current_timestamp() AS processed_at,
    COUNT(*) AS records_processed
FROM sales_incremental
GROUP BY _source_file;

num_affected_rows,num_inserted_rows
1,1
